# 🟠 Google Data Engineer – Page 3 (Q711-Q1111)
| # | Title | Category | Difficulty |
|---|-------|----------|-----------|
| Q711 | Symmetrize Friends Network | SQL | MEDIUM |
| Q775 | Top Visited Website | SQL | HARD |
| Q792 | Longest Winning Run | SQL | MEDIUM |
| Q829 | Subarray Sum Equals K | Python | MEDIUM |
| Q837 | Google Maps Trips | SQL | HARD |
| Q1069 | Email Activity Ranking | SQL | MEDIUM |
| Q1103 | CTR by Ad Type | SQL | MEDIUM |


---
## Q711 · Symmetrize Friends Network (Database – MEDIUM)

### Problem
Given a `friends` table (user1, user2), some edges appear once-directional.
Symmetrize it: ensure if (A,B) exists, (B,A) also exists. Return unique pairs.

---
### 🧠 How to Remember
```
1. UNION original table with SWAPPED columns
2. Use UNION (not UNION ALL) to deduplicate
3. Or: add WHERE user1 < user2 to get canonical pairs
```


In [ ]:
import sqlite3
import pandas as pd

conn3 = sqlite3.connect(":memory:")
conn3.execute("CREATE TABLE friends (user1 INT, user2 INT)")
conn3.executemany("INSERT INTO friends VALUES (?,?)", [
    (1, 2), (1, 3), (2, 3), (3, 4), (4, 1)
])

# Solution 1: UNION to symmetrize
sql_sym1 = """
SELECT user1, user2 FROM friends
UNION
SELECT user2 AS user1, user1 AS user2 FROM friends
ORDER BY user1, user2
"""
print("Solution 1 - UNION symmetrize:")
print(pd.read_sql(sql_sym1, conn3))


In [ ]:
# Solution 2: Canonical pairs (user1 < user2)
sql_sym2 = """
SELECT DISTINCT
    MIN(user1, user2) AS user_a,
    MAX(user1, user2) AS user_b
FROM (
    SELECT user1, user2 FROM friends
    UNION ALL
    SELECT user2, user1 FROM friends
)
ORDER BY user_a, user_b
"""
print("Solution 2 - Canonical pairs (deduped):")
print(pd.read_sql(sql_sym2, conn3))


In [ ]:
# Solution 3: With mutual friend count
sql_sym3 = """
WITH all_edges AS (
    SELECT user1, user2 FROM friends
    UNION
    SELECT user2, user1 FROM friends
)
SELECT a.user1, a.user2,
       COUNT(DISTINCT m.user2) AS mutual_friends
FROM all_edges a
LEFT JOIN all_edges m
    ON m.user1 = a.user1
    AND m.user2 IN (SELECT user2 FROM all_edges WHERE user1 = a.user2)
GROUP BY a.user1, a.user2
ORDER BY a.user1, a.user2
"""
print("Solution 3 - With mutual friends count:")
print(pd.read_sql(sql_sym3, conn3))


---
## Q792 · Longest Winning Run (Database – MEDIUM)

### Problem
Given `game_results` (player_id, game_date, win INT), find the longest consecutive
winning streak for each player.

---
### 🧠 How to Remember
```
GAP-AND-ISLANDS technique:
1. ROW_NUMBER() overall
2. ROW_NUMBER() partitioned by player+win
3. Difference = "island id" for consecutive same-results
4. COUNT of consecutive wins per island → MAX per player
```


In [ ]:
conn3.execute("""
CREATE TABLE game_results (player_id INT, game_date TEXT, win INT)
""")
conn3.executemany("INSERT INTO game_results VALUES (?,?,?)", [
    (1, '2023-01-01', 1),
    (1, '2023-01-02', 1),
    (1, '2023-01-03', 0),
    (1, '2023-01-04', 1),
    (1, '2023-01-05', 1),
    (1, '2023-01-06', 1),
    (2, '2023-01-01', 1),
    (2, '2023-01-02', 0),
    (2, '2023-01-03', 1),
])

# Solution 1: Gap-and-islands with ROW_NUMBER
sql_win1 = """
WITH numbered AS (
    SELECT player_id, game_date, win,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY game_date) AS rn,
           ROW_NUMBER() OVER (PARTITION BY player_id, win ORDER BY game_date) AS rn_win
    FROM game_results
),
islands AS (
    SELECT player_id, win, (rn - rn_win) AS island_id, COUNT(*) AS streak_len
    FROM numbered
    WHERE win = 1
    GROUP BY player_id, win, island_id
)
SELECT player_id, MAX(streak_len) AS longest_winning_streak
FROM islands
GROUP BY player_id
ORDER BY player_id
"""
print("Solution 1 - Gap-and-Islands:")
print(pd.read_sql(sql_win1, conn3))


In [ ]:
# Solution 2: Self-join (classic approach)
sql_win2 = """
WITH wins_only AS (
    SELECT player_id, game_date,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY game_date) AS rn
    FROM game_results WHERE win = 1
),
grouped AS (
    SELECT player_id, COUNT(*) AS streak,
           date(game_date, '-' || (rn-1) || ' days') AS group_key
    FROM wins_only
    GROUP BY player_id, group_key
)
SELECT player_id, MAX(streak) AS longest_winning_streak
FROM grouped
GROUP BY player_id
ORDER BY player_id
"""
print("Solution 2 - Date-based grouping:")
print(pd.read_sql(sql_win2, conn3))


In [ ]:
# Solution 3: Python simulation (when SQL isn't available)
# Time: O(n log n)  Space: O(n)
from itertools import groupby

def longest_winning_run(records):
    """records: list of (player_id, date, win)"""
    from collections import defaultdict
    player_games = defaultdict(list)
    for pid, date, win in records:
        player_games[pid].append((date, win))

    result = {}
    for pid, games in player_games.items():
        games.sort()
        max_streak = 0
        curr_streak = 0
        for _, win in games:
            if win == 1:
                curr_streak += 1
                max_streak = max(max_streak, curr_streak)
            else:
                curr_streak = 0
        result[pid] = max_streak
    return result

data = [(1,'2023-01-01',1),(1,'2023-01-02',1),(1,'2023-01-03',0),
        (1,'2023-01-04',1),(1,'2023-01-05',1),(1,'2023-01-06',1),
        (2,'2023-01-01',1),(2,'2023-01-02',0),(2,'2023-01-03',1)]
print("Solution 3 - Python:", longest_winning_run(data))


---
## Q829 · Subarray Sum Equals K (Data Structures – MEDIUM)

### Problem
Given array nums and integer k, count subarrays that sum to exactly k.

**Example:** nums=[1,1,1], k=2 → 2  (subarrays [1,1] at positions 0-1 and 1-2)

---
### 🧠 How to Remember
```
PREFIX SUM + HASH MAP trick:
1. prefix_sum[i] = sum of nums[0..i]
2. subarray sum from j+1 to i = prefix[i] - prefix[j]
3. We want prefix[i] - prefix[j] = k  →  prefix[j] = prefix[i] - k
4. Count how many times (prefix[i] - k) appeared before!
MANTRA: "If I've seen prefix_sum - k before, I found a valid subarray"
```


In [ ]:
# Solution 1: Brute Force - check all subarrays
# Time: O(n²)  Space: O(1)
def subarray_sum_brute(nums: List[int], k: int) -> int:
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            if total == k:
                count += 1
    return count

print("Solution 1:", subarray_sum_brute([1,1,1], 2))  # 2
print("Solution 1:", subarray_sum_brute([1,2,3], 3))   # 2


In [ ]:
# Solution 2: Prefix Sum array
# Time: O(n²)  Space: O(n)
def subarray_sum_prefix(nums: List[int], k: int) -> int:
    n = len(nums)
    prefix = [0] * (n + 1)
    for i in range(n):
        prefix[i+1] = prefix[i] + nums[i]

    count = 0
    for i in range(n+1):
        for j in range(i+1, n+1):
            if prefix[j] - prefix[i] == k:
                count += 1
    return count

print("Solution 2:", subarray_sum_prefix([1,1,1], 2))  # 2


In [ ]:
# Solution 3: HashMap of prefix sums ← OPTIMAL
# Time: O(n)  Space: O(n)
from collections import defaultdict

def subarray_sum_hashmap(nums: List[int], k: int) -> int:
    """
    Track frequency of prefix sums seen so far.
    If (current_prefix - k) was seen before → valid subarray found!
    """
    count = 0
    prefix = 0
    freq = defaultdict(int)
    freq[0] = 1       # empty subarray has sum 0

    for num in nums:
        prefix += num
        count += freq[prefix - k]   # how many j's satisfy sum[j..i] = k
        freq[prefix] += 1

    return count

print("Solution 3:", subarray_sum_hashmap([1,1,1], 2))   # 2
print("Solution 3:", subarray_sum_hashmap([1,2,3], 3))    # 2
print("Solution 3:", subarray_sum_hashmap([-1,-1,1], 0))  # 1
print("\n⏱ Complexity Summary:")
print("| Solution    | Time  | Space |")
print("|-------------|-------|-------|")
print("| Brute       | O(n²) | O(1)  |")
print("| Prefix Array| O(n²) | O(n)  |")
print("| HashMap ✅  | O(n)  | O(n)  |")


---
## Q1069 · Email Activity Ranking (Database – MEDIUM)

### Problem
Given `emails` (from_user, to_user, day) count emails sent AND received per user,
then rank users by total activity (sent + received).

---
### 🧠 How to Remember
```
1. Two CTEs: sent (GROUP BY from_user), received (GROUP BY to_user)
2. FULL OUTER JOIN (use UNION in SQLite)
3. total = sent + received, RANK() by total
```


In [ ]:
conn3.execute("CREATE TABLE email_log (from_user TEXT, to_user TEXT, day INT)")
conn3.executemany("INSERT INTO email_log VALUES (?,?,?)", [
    ('Alice','Bob',1), ('Alice','Carol',1), ('Bob','Alice',2),
    ('Carol','Alice',3), ('Carol','Bob',3), ('Bob','Carol',4),
    ('Alice','Bob',5),
])

# Solution 1: UNION-based FULL OUTER JOIN equivalent
sql_email_rank1 = """
WITH sent AS (
    SELECT from_user AS user, COUNT(*) AS sent_count
    FROM email_log GROUP BY from_user
),
recv AS (
    SELECT to_user AS user, COUNT(*) AS recv_count
    FROM email_log GROUP BY to_user
),
all_users AS (
    SELECT user FROM sent UNION SELECT user FROM recv
)
SELECT a.user,
       COALESCE(s.sent_count, 0) AS sent,
       COALESCE(r.recv_count, 0) AS received,
       COALESCE(s.sent_count,0) + COALESCE(r.recv_count,0) AS total_activity
FROM all_users a
LEFT JOIN sent s ON a.user = s.user
LEFT JOIN recv r ON a.user = r.user
ORDER BY total_activity DESC
"""
print("Solution 1 - Email activity:")
print(pd.read_sql(sql_email_rank1, conn3))


In [ ]:
# Solution 2: UNION ALL + GROUP BY (more concise)
sql_email_rank2 = """
WITH activity AS (
    SELECT from_user AS user, 'sent' AS type FROM email_log
    UNION ALL
    SELECT to_user AS user, 'received' AS type FROM email_log
)
SELECT user,
       SUM(CASE WHEN type='sent' THEN 1 ELSE 0 END) AS sent,
       SUM(CASE WHEN type='received' THEN 1 ELSE 0 END) AS received,
       COUNT(*) AS total_activity,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS activity_rank
FROM activity
GROUP BY user
ORDER BY total_activity DESC
"""
print("Solution 2 - UNION ALL approach:")
print(pd.read_sql(sql_email_rank2, conn3))


In [ ]:
# Solution 3: Pandas equivalent (Python)
import pandas as pd as pd_lib
df = pd.read_sql("SELECT * FROM email_log", conn3)
sent = df.groupby('from_user').size().rename('sent')
recv = df.groupby('to_user').size().rename('received')
combined = pd.concat([sent, recv], axis=1).fillna(0).astype(int)
combined['total'] = combined['sent'] + combined['received']
combined = combined.sort_values('total', ascending=False)
combined['rank'] = combined['total'].rank(method='min', ascending=False).astype(int)
print("Solution 3 - Pandas:")
print(combined)
